In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_groq import ChatGroq

In [8]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [12]:
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0)
def generate_joke(state: JokeState):
    prompt = f"Generate a joke on the topic: {state['topic']}"
    response = llm.invoke(prompt).content
    return {"joke": response}

def generate_explanation(state: JokeState):
    prompt = f"Right an explanation for the joke: {state['joke']}"
    response = llm.invoke(prompt).content
    return {"explanation": response}

In [13]:
# Create the state graph
graph = StateGraph(JokeState)

# Add nodes
graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

# Add edges
graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)

# Instantiate the checkpointer for persistence
checkpointer = MemorySaver()

# Compile graph with the checkpointer registered
workflow = graph.compile(checkpointer=checkpointer)

In [14]:
# Configuration block carrying the specific session identifier
config = {"configurable": {"thread_id": "1"}}

# Initial invocation pass
initial_state = {"topic": "pizza"}
result = workflow.invoke(initial_state, config=config)

In [16]:
print(result)

{'topic': 'pizza', 'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!", 'explanation': 'The joke!\n\nThe joke is a play on words. The phrase "feeling crusty" has a double meaning here.\n\nIn one sense, a pizza has a crust, which is the outer layer of the dough that forms the base of the pizza.\n\nBut the phrase "feeling crusty" also sounds similar to "feeling cranky" or "feeling grumpy", which means being in a bad mood.\n\nSo, the joke is saying that the pizza is in a bad mood (crusty as in grumpy), and it\'s making a pun on the fact that a pizza literally has a crust. It\'s a lighthearted and cheesy (pun intended) joke that\'s meant to make you laugh!'}


In [17]:
# Fetching the most recent final state snapshot
final_state = workflow.get_state(config)
print(final_state.values)

# Fetching the entire history log of intermediate checkpoints
history = list(workflow.get_state_history(config))
for state_snapshot in history:
    print("Checkpoint ID:", state_snapshot.config["configurable"]["checkpoint_id"])
    print("Values:", state_snapshot.values)
    print("Next Node to Execute:", state_snapshot.next)
    print("-" * 50)

{'topic': 'pizza', 'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!", 'explanation': 'The joke!\n\nThe joke is a play on words. The phrase "feeling crusty" has a double meaning here.\n\nIn one sense, a pizza has a crust, which is the outer layer of the dough that forms the base of the pizza.\n\nBut the phrase "feeling crusty" also sounds similar to "feeling cranky" or "feeling grumpy", which means being in a bad mood.\n\nSo, the joke is saying that the pizza is in a bad mood (crusty as in grumpy), and it\'s making a pun on the fact that a pizza literally has a crust. It\'s a lighthearted and cheesy (pun intended) joke that\'s meant to make you laugh!'}
Checkpoint ID: 1f16d6a5-29ea-6c0c-8002-3bb72a68476a
Values: {'topic': 'pizza', 'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!", 'explanation': 'The joke!\n\nThe joke is a play on words. The phrase "feeling crusty" has a double meaning 

In [20]:
# =====================================================================
# TIME TRAVEL PART: FETCHING HISTORY, MODIFYING STATE, AND FORKING
# =====================================================================

# 1. Fetch the entire history log of intermediate checkpoints for the thread
# This prints all past snapshots along with their unique checkpoint_id
history = list(workflow.get_state_history(config))
for state_snapshot in history:
    print("Checkpoint ID:", state_snapshot.config["configurable"]["checkpoint_id"])
    print("Values:", state_snapshot.values)
    print("Next Node to Execute:", state_snapshot.next)
    print("-" * 50)

# 2. Target a specific past milestone checkpoint ID from your printed list
# (e.g., targeting the snapshot right after the topic was given but before the joke ran)
target_checkpoint_id = "1f16d6a5-25e3-610c-8001-7053ec6c8193" 

# 3. Update the state at that specific past milestone to fork a new branch.
# Note: Including 'checkpoint_ns': '' prevents the KeyError you encountered.
workflow.update_state(
    {
        "configurable": {
            "thread_id": "1", 
            "checkpoint_id": target_checkpoint_id,
            "checkpoint_ns": ""  # Explicitly defined to fulfill the namespace requirement
        }
    },
    {"topic": "samosa"},       # Overriding the original topic ('pizza' -> 'samosa')
    as_node="generate_joke"    # Simulating that this update originated from this node
)

# 4. Retrieve the newly updated history to catch the newly created fork's head ID
branched_history = list(workflow.get_state_history(config))
new_fork_id = branched_history[0].config["configurable"]["checkpoint_id"]

# 5. Resume and execute control down the fresh time-traveled branch path
fork_config = {
    "configurable": {
        "thread_id": "1", 
        "checkpoint_id": new_fork_id,
        "checkpoint_ns": ""
    }
}

# Invoke with None to let it continue execution from the targeted fork point
branched_output = workflow.invoke(None, config=fork_config)
print("Branched State Output:", branched_output)

Checkpoint ID: 1f16d6bb-7e61-6eab-8000-add224d78034
Values: {'topic': 'samosa'}
Next Node to Execute: ('generate_explanation',)
--------------------------------------------------
Checkpoint ID: 1f16d6a5-29ea-6c0c-8002-3bb72a68476a
Values: {'topic': 'pizza', 'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!", 'explanation': 'The joke!\n\nThe joke is a play on words. The phrase "feeling crusty" has a double meaning here.\n\nIn one sense, a pizza has a crust, which is the outer layer of the dough that forms the base of the pizza.\n\nBut the phrase "feeling crusty" also sounds similar to "feeling cranky" or "feeling grumpy", which means being in a bad mood.\n\nSo, the joke is saying that the pizza is in a bad mood (crusty as in grumpy), and it\'s making a pun on the fact that a pizza literally has a crust. It\'s a lighthearted and cheesy (pun intended) joke that\'s meant to make you laugh!'}
Next Node to Execute: ()
------------------------